In [ ]:
import pandas as pd
from pathlib import Path
import re
import time

# ==================================================
# FOLDER NAME
# Put all raw CSV files inside this folder
# ==================================================

folder = Path("degree_year")

# Output file with ALL original columns + clean columns
output_file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

# ==================================================
# AWARD LEVEL MAP
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

# ==================================================
# DEGREE GROUP
# ==================================================

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    x = int(x)

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [10, 11]:
        return "Professional"
    elif x in [1, 2, 4, 6, 8]:
        return "Certificate"
    else:
        return "Other"

# ==================================================
# CLEAN CIP CODE
# Example:
# 10102  -> 01.0102
# 6.0101 -> 06.0101
# ==================================================

def clean_cip_code(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().replace('"', "")

    if s == "" or s.lower() in ["nan", "none"]:
        return pd.NA

    if "." in s:
        left, right = s.split(".", 1)
        left = re.sub(r"\D", "", left).zfill(2)
        right = re.sub(r"\D", "", right).ljust(4, "0")[:4]
        return f"{left}.{right}"

    digits = re.sub(r"\D", "", s)

    if digits == "":
        return pd.NA

    digits = digits.zfill(6)[-6:]
    return f"{digits[:2]}.{digits[2:]}"

# ==================================================
# GET YEAR FROM FILE NAME
# Example:
# 1984_c1984_cip.csv -> 1984
# ==================================================

def extract_year(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group(0))
    return pd.NA

# ==================================================
# READ CSV SAFELY
# ==================================================

def read_csv_safely(file_path):
    try:
        return pd.read_csv(file_path, dtype=str, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, dtype=str, low_memory=False, encoding="latin1")

# ==================================================
# CHECK FOLDER
# ==================================================

print("Current Python folder:")
print(Path.cwd())

print("\nLooking for CSV files inside:")
print(folder.resolve())

if not folder.exists():
    raise FileNotFoundError(
        f"Folder not found: {folder}\n"
        "Create a folder named degree_year and put your CSV files inside it."
    )

files = sorted(folder.glob("*.csv"))

if len(files) == 0:
    raise FileNotFoundError(
        "No CSV files found inside degree_year.\n"
        "Make sure your files like 1984_c1984_cip.csv are inside the degree_year folder."
    )

print(f"\nFound {len(files)} CSV files.")

print("\nFirst few files:")
for f in files[:10]:
    print(f.name)

# ==================================================
# READ ALL FILES AND KEEP ALL ORIGINAL COLUMNS
# ==================================================

all_data = []
errors = []

start_time = time.time()

for i, file_path in enumerate(files, start=1):
    try:
        df = read_csv_safely(file_path)

        # Clean column names
        df.columns = df.columns.str.lower().str.strip()

        # Add year and source file
        df["year"] = extract_year(file_path.name)
        df["source_file"] = file_path.name

        # Add clean CIP code
        if "cipcode" in df.columns:
            df["cip_code_clean"] = df["cipcode"].apply(clean_cip_code)
        else:
            df["cip_code_clean"] = pd.NA

        # Add clean award level
        if "awlevel" in df.columns:
            df["awlevel_clean"] = pd.to_numeric(
                df["awlevel"],
                errors="coerce"
            ).astype("Int64")

            df["degree_level"] = df["awlevel_clean"].map(award_level_map).fillna("Unknown")
            df["degree_group"] = df["awlevel_clean"].apply(degree_group)
        else:
            df["awlevel_clean"] = pd.NA
            df["degree_level"] = "Unknown"
            df["degree_group"] = "Unknown"

        # Add degree count
        # crace15 = total men
        # crace16 = total women
        if "crace15" in df.columns and "crace16" in df.columns:
            df["men_count"] = pd.to_numeric(df["crace15"], errors="coerce").fillna(0).astype(int)
            df["women_count"] = pd.to_numeric(df["crace16"], errors="coerce").fillna(0).astype(int)
            df["degree_count"] = df["men_count"] + df["women_count"]
        else:
            df["men_count"] = 0
            df["women_count"] = 0
            df["degree_count"] = 0

        # Keep ALL original columns + new clean columns
        all_data.append(df)

    except Exception as e:
        errors.append({
            "file": file_path.name,
            "error": str(e)
        })

    elapsed = time.time() - start_time
    avg_time = elapsed / i
    remaining = avg_time * (len(files) - i)

    print(
        f"Loading: {i}/{len(files)} "
        f"({i / len(files):.0%}) | "
        f"Elapsed: {elapsed:.1f}s | "
        f"Left: {remaining:.1f}s",
        end="\r"
    )

print("\n\nFinished reading all files.")

# ==================================================
# COMBINE EVERYTHING
# ==================================================

if len(all_data) == 0:
    raise ValueError("No data was loaded. Check the errors above.")

all_info = pd.concat(all_data, ignore_index=True, sort=False)

# ==================================================
# OPTIONAL SORT
# ==================================================

sort_cols = []

for col in ["year", "unitid", "cip_code_clean", "awlevel_clean"]:
    if col in all_info.columns:
        sort_cols.append(col)

if len(sort_cols) > 0:
    all_info = all_info.sort_values(by=sort_cols).reset_index(drop=True)

# ==================================================
# SAVE FINAL FILE
# ==================================================

all_info.to_csv(output_file, index=False)

# ==================================================
# PRINT RESULTS
# ==================================================

print("\nSaved file:")
print(output_file)

print("\nTotal rows and columns:")
print(all_info.shape)

print("\nTotal rows:")
print(len(all_info))

print("\nTotal columns:")
print(len(all_info.columns))

print("\nAll column names:")
for col in all_info.columns:
    print(col)

print("\nPreview first 10 rows:")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
print(all_info.head(10))

# ==================================================
# PRINT ERRORS IF ANY
# ==================================================

if errors:
    print("\nFiles with errors:")
    for e in errors:
        print(e["file"], "->", e["error"])
else:
    print("\nNo file errors.")

# ==================================================
# HOW TO READ THE SAVED FILE LATER
# ==================================================

print("\nLater, read the clean file with:")
print('df = pd.read_csv("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv", low_memory=False)')

In [ ]:
df = pd.read_csv("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv", low_memory=False)

In [ ]:
import pandas as pd
from pathlib import Path
import re
import time
import sys

# ==================================================
# FOLDER NAME
# ==================================================

folder = Path("degree_year")

# Output file with ALL original columns + clean columns
output_file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

# ==================================================
# AWARD LEVEL MAP
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

# ==================================================
# DEGREE GROUP FUNCTION
# ==================================================

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    try:
        x = int(float(x))
    except:
        return "Unknown"

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [1, 2, 4, 6, 8, 10, 11]:
        return "Certificate / Other"
    else:
        return "Unknown"

# ==================================================
# CLEAN TEXT FUNCTION
# ==================================================

def clean_text(value):
    """
    Cleans text safely.
    Prevents real NaN from becoming the string 'nan'.
    """
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null"]:
        return ""

    value = re.sub(r"\s+", " ", value)

    return value

# ==================================================
# CLEAN NUMBER FUNCTION
# ==================================================

def clean_number(value):
    """
    Converts number safely.
    Missing or bad values become 0.
    """
    if pd.isna(value):
        return 0

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null", ""]:
        return 0

    value = value.replace(",", "")

    return pd.to_numeric(value, errors="coerce")

# ==================================================
# FIND COLUMN SAFELY
# ==================================================

def find_col(df, possible_names):
    """
    Finds the first matching column name from a list.
    Works even if column names have lowercase/uppercase differences.
    """
    col_map = {col.upper(): col for col in df.columns}

    for name in possible_names:
        if name.upper() in col_map:
            return col_map[name.upper()]

    return None

# ==================================================
# GET YEAR FROM FILE NAME
# ==================================================

def get_year_from_filename(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group())
    return None

# ==================================================
# PROGRESS BAR
# ==================================================

def show_progress(current, total, start_time, filename=""):
    percent = current / total
    bar_length = 30
    filled_length = int(bar_length * percent)

    bar = "█" * filled_length + "-" * (bar_length - filled_length)

    elapsed = time.time() - start_time

    if current > 0:
        estimated_total = elapsed / current * total
        remaining = estimated_total - elapsed
    else:
        remaining = 0

    sys.stdout.write(
        f"\rLoading: |{bar}| {current}/{total} "
        f"({percent:.1%}) | Elapsed: {elapsed:.1f}s | Left: {remaining:.1f}s | {filename[:35]}"
    )
    sys.stdout.flush()

# ==================================================
# CHECK FOLDER
# ==================================================

if not folder.exists():
    raise FileNotFoundError(f"Folder not found: {folder}")

csv_files = sorted(folder.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found inside folder: {folder}")

print(f"Found {len(csv_files)} CSV files.")
print("Starting clean and combine...\n")

# ==================================================
# READ + CLEAN + COMBINE
# ==================================================

all_data = []
start_time = time.time()

for i, file in enumerate(csv_files, start=1):

    show_progress(i, len(csv_files), start_time, file.name)

    try:
        df = pd.read_csv(
            file,
            dtype=str,
            low_memory=False,
            encoding="latin1"
        )

    except Exception as e:
        print(f"\nSkipping file because of error: {file.name}")
        print(e)
        continue

    # ----------------------------------------------
    # Add year from filename
    # ----------------------------------------------

    year = get_year_from_filename(file.name)
    df["clean_year"] = year

    # ----------------------------------------------
    # Find useful IPEDS columns safely
    # ----------------------------------------------

    unitid_col = find_col(df, ["UNITID"])
    cip_col = find_col(df, ["CIPCODE", "CIP", "CIP_CODE"])
    major_col = find_col(df, ["CIPTITLE", "CIPDESC", "MAJOR_NAME", "PROGRAM"])
    awlevel_col = find_col(df, ["AWLEVEL", "AWARD_LEVEL"])
    count_col = find_col(df, ["CTOTALT", "XCTOTALT", "TOTAL", "COUNT"])

    # ----------------------------------------------
    # Clean UNITID
    # ----------------------------------------------

    if unitid_col:
        df["clean_unitid"] = df[unitid_col].apply(clean_text)
    else:
        df["clean_unitid"] = ""

    # ----------------------------------------------
    # Clean CIP code
    # ----------------------------------------------

    if cip_col:
        df["clean_cip_code"] = df[cip_col].apply(clean_text)
    else:
        df["clean_cip_code"] = ""

    # ----------------------------------------------
    # Clean major/program name
    # This prevents "nan" from showing
    # ----------------------------------------------

    if major_col:
        df["clean_major_name"] = df[major_col].apply(clean_text)
    else:
        df["clean_major_name"] = ""

    # ----------------------------------------------
    # Clean award level
    # ----------------------------------------------

    if awlevel_col:
        df["clean_awlevel_code"] = pd.to_numeric(df[awlevel_col], errors="coerce")
        df["clean_award_level"] = df["clean_awlevel_code"].map(award_level_map)
        df["clean_degree_group"] = df["clean_awlevel_code"].apply(degree_group)
    else:
        df["clean_awlevel_code"] = pd.NA
        df["clean_award_level"] = "Unknown"
        df["clean_degree_group"] = "Unknown"

    df["clean_award_level"] = df["clean_award_level"].fillna("Unknown")
    df["clean_degree_group"] = df["clean_degree_group"].fillna("Unknown")

    # ----------------------------------------------
    # Clean degree count
    # ----------------------------------------------

    if count_col:
        df["clean_degree_count"] = df[count_col].apply(clean_number)
    else:
        df["clean_degree_count"] = 0

    df["clean_degree_count"] = pd.to_numeric(
        df["clean_degree_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    # ----------------------------------------------
    # Final cleanup: remove NaN text from object columns
    # ----------------------------------------------

    text_cols = df.select_dtypes(include=["object"]).columns

    for col in text_cols:
        df[col] = df[col].fillna("")
        df[col] = df[col].replace(["nan", "NaN", "None", "NONE", "null", "NULL"], "")

    all_data.append(df)

print("\n\nCombining all files...")

# ==================================================
# COMBINE ALL FILES
# ==================================================

final_dataset = pd.concat(all_data, ignore_index=True)

# ==================================================
# FINAL CLEANUP
# ==================================================

text_cols = final_dataset.select_dtypes(include=["object"]).columns

for col in text_cols:
    final_dataset[col] = final_dataset[col].fillna("")
    final_dataset[col] = final_dataset[col].replace(
        ["nan", "NaN", "None", "NONE", "null", "NULL"],
        ""
    )

# ==================================================
# SAVE FILE
# ==================================================

final_dataset.to_csv(output_file, index=False)

total_time = time.time() - start_time

print("\nDone!")
print(f"Saved file: {output_file}")
print(f"Rows: {len(final_dataset):,}")
print(f"Columns: {len(final_dataset.columns):,}")
print(f"Total time: {total_time:.2f} seconds")